# Depth Estimation Error Analysis

This notebook analyzes the errors between actual depth and predicted depth (depth_median) from the flower detection results.

## Section 1: Load and Explore the CSV Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load the CSV
df = pd.read_csv('flower_detection_results_annotated.csv')

# Display basic information
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics for Actual_Depth and Depth_Median:")
print(df[['Actual_Depth', 'Depth_Median']].describe())

## Section 2: Calculate Error Metrics

In [ ]:
# Calculate errors
df['Error'] = df['Depth_Median'] - df['Actual_Depth']  # Prediction - Actual
df['Abs_Error'] = np.abs(df['Error'])
df['Percent_Error'] = (df['Abs_Error'] / df['Actual_Depth']) * 100

# Calculate error metrics
mae = df['Abs_Error'].mean()
rmse = np.sqrt(np.mean(df['Error']**2))
mape = df['Percent_Error'].mean()

print(f"Mean Absolute Error (MAE): {mae:.4f} m")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} m")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"\nError statistics:")
print(df[['Error', 'Abs_Error', 'Percent_Error']].describe())

# Error by baseline
print("\n\nError by Baseline:")
for baseline in df['Baseline'].unique():
    subset = df[df['Baseline'] == baseline]
    mae_b = subset['Abs_Error'].mean()
    rmse_b = np.sqrt(np.mean(subset['Error']**2))
    print(f"  {baseline}: MAE={mae_b:.4f}m, RMSE={rmse_b:.4f}m")

## Section 3: Scatter Plot - Actual vs Predicted Depth

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Create color map for different baselines
baselines = df['Baseline'].unique()
colors = {'16_v1': '#1f77b4', '16_v2': '#ff7f0e', '11': '#2ca02c'}

for baseline in baselines:
    subset = df[df['Baseline'] == baseline]
    ax.scatter(subset['Actual_Depth'], subset['Depth_Median'], 
               label=baseline, alpha=0.6, s=50, color=colors[baseline])

# Add perfect prediction line (diagonal)
min_val = min(df['Actual_Depth'].min(), df['Depth_Median'].min())
max_val = max(df['Actual_Depth'].max(), df['Depth_Median'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', label='Perfect prediction', linewidth=2)

ax.set_xlabel('Actual Depth (m)', fontsize=12)
ax.set_ylabel('Predicted Depth - Depth_Median (m)', fontsize=12)
ax.set_title('Actual vs Predicted Depth', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 4: Residual Plot - Error Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for baseline in baselines:
    subset = df[df['Baseline'] == baseline]
    ax.scatter(subset['Depth_Median'], subset['Error'], 
               label=baseline, alpha=0.6, s=50, color=colors[baseline])

# Add reference line at zero error
ax.axhline(y=0, color='k', linestyle='--', linewidth=2, label='Zero error')

ax.set_xlabel('Predicted Depth - Depth_Median (m)', fontsize=12)
ax.set_ylabel('Error = Predicted - Actual (m)', fontsize=12)
ax.set_title('Residual Plot: Error vs Predicted Depth', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 5: Histogram - Error Magnitude

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(df['Abs_Error'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
ax.axvline(mae, color='red', linestyle='--', linewidth=2, label=f'Mean: {mae:.4f} m')
ax.axvline(df['Abs_Error'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["Abs_Error"].median():.4f} m')

ax.set_xlabel('Absolute Error (m)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Absolute Error Magnitude', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Section 6: Box Plot - Error Statistics by Baseline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot for absolute error by baseline
sns.boxplot(data=df, x='Baseline', y='Abs_Error', ax=axes[0], palette=[colors[b] for b in df['Baseline'].unique()])
axes[0].set_ylabel('Absolute Error (m)', fontsize=11)
axes[0].set_xlabel('Baseline', fontsize=11)
axes[0].set_title('Absolute Error Distribution by Baseline', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Box plot for signed error by baseline
sns.boxplot(data=df, x='Baseline', y='Error', ax=axes[1], palette=[colors[b] for b in df['Baseline'].unique()])
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_ylabel('Signed Error (m)', fontsize=11)
axes[1].set_xlabel('Baseline', fontsize=11)
axes[1].set_title('Signed Error Distribution by Baseline', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Section 7: Error by Position and Baseline

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Create a combined label for baseline and position
df['Baseline_Pos'] = df['Baseline'] + '_Pos' + df['Position'].astype(str)

# Plot
sns.boxplot(data=df, x='Baseline_Pos', y='Abs_Error', ax=ax)
ax.set_ylabel('Absolute Error (m)', fontsize=12)
ax.set_xlabel('Baseline and Position', fontsize=12)
ax.set_title('Absolute Error by Baseline and Position', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Print detailed stats
print("Error Statistics by Baseline and Position:")
print("=" * 80)
for baseline in sorted(df['Baseline'].unique()):
    for pos in sorted(df[df['Baseline']==baseline]['Position'].unique()):
        subset = df[(df['Baseline']==baseline) & (df['Position']==pos)]
        print(f"\n{baseline} - Position {pos} (Actual: {subset['Actual_Depth'].iloc[0]:.3f}m):")
        print(f"  Predicted: {subset['Depth_Median'].mean():.4f} ± {subset['Depth_Median'].std():.4f}")
        print(f"  MAE: {subset['Abs_Error'].mean():.4f} m")
        print(f"  RMSE: {np.sqrt(np.mean(subset['Error']**2)):.4f} m")

## Section 8: Summary Statistics and Insights